# Cleaning Lyrics Data

This lyrics data cleaning used to correct the wrong lyrics pulled in by Lyrics Genius API. This happened because their is inconsistency in the way some track name and artist name appear in Lyrics Genius and how they appear in Spotify. Two functions were created for cleaning and another two for pulling the lyrics. Two to check for wrong lyrics by counting the number of times the track name appear in the lyrics and the other to check if it contains data and the word "feat". This was the most stressful and time consuming part of my work, because even after cleaning I had to maually do it myself.

# Load Datasets

This dataset were loaded from the dataset in Lyrics and Musical Feature Data Collection Notebook

In [2]:
import re
import pandas as pd
import lyricsgenius as genius

gym = pd.read_csv('/Users/mac/Downloads/Bar.csv')
bedroom = pd.read_csv('/Users/mac/Downloads/Bedroom.csv') 
library = pd.read_csv('/Users/mac/Downloads/Library.csv') 
office = pd.read_csv('/Users/mac/Downloads/Office.csv') 
trainBus = pd.read_csv('/Users/mac/Downloads/TrainBus.csv')
restaurant = pd.read_csv('/Users/mac/Downloads/Restaurant.csv')
bar = pd.read_csv('/Users/mac/Downloads/Bar.csv')
mall = pd.read_csv('/Users/mac/Downloads/Mall.csv')

# Clean Lyrics

In [80]:


api=genius.Genius('lwGgNIQ7ObjCzE1uuDjxOyX5IgUVlcfZBDdG2CAKcBsehFodJiivZz_T23dfrs4J')

def count_title_in_lyrics(track_name, lyrics):
    """
    Count the number of times the cleaned track_name appears in the lyrics.
    """
    # Clean the track name
    cleaned_track_name = clean_track_name(track_name)
    
    # Convert lyrics to lowercase for case-insensitive comparison
    lyrics_lower = lyrics.lower()
    
    # Count the occurrences of the cleaned track name in the lyrics
    count = lyrics_lower.count(cleaned_track_name)
    
    return count

pattern = re.compile(r"\b\d{2}/\d{2}/\d{2}\b|feat", re.IGNORECASE)

def match_pattern(lyrics):
    return bool(pattern.search(lyrics))


# Function to clean track names by removing remaster suffixes and text within parentheses
def clean_track_name(track_name):
    # First, remove any text within parentheses, including the parentheses themselves
    cleaned_name = re.sub(r'\s*\(.*?\)', '', track_name)
    # Then, remove any " - Year Remaster" or similar suffixes
    cleaned_name = re.sub(r' - \d{4}.*$', '', cleaned_name)
    # Finally, strip any leading or trailing whitespace
    return cleaned_name.strip()

def fetch_lyrics_in_batches(api, df, track_col='track_name', artist_col='artist_name', batch_size=100):
    all_lyrics = []

    # Clean the track names in the DataFrame
    df['cleaned_track_name'] = df[track_col].apply(clean_track_name)

    # Split the track names and artists into batches of up to 100
    tracks = df['cleaned_track_name'].tolist()
    artists = df[artist_col].tolist()

    for i in range(0, len(tracks), batch_size):
        batch_tracks = tracks[i:i + batch_size]
        batch_artists = artists[i:i + batch_size]
        batch_lyrics = []

        for track, artist in zip(batch_tracks, batch_artists):
            try:
                song = api.search_song(title=track, artist=artist)
                if song:
                    lyric = song.lyrics
                    batch_lyrics.append(lyric)
                else:
                    batch_lyrics.append('Lyrics not found')
            except Exception as e:
                print(f'Error fetching lyrics for {track} by {artist}: {str(e)}')
                batch_lyrics.append(f'Error: {str(e)}')

        all_lyrics.extend(batch_lyrics)

    # Append lyrics to the DataFrame
    df['lyrics'] = all_lyrics
    return df

In [185]:
def clean_track_name(track_name):
    """
    Clean the track name by removing any text within parentheses and any suffixes starting with '-'.
    """
    # Remove text within parentheses, including the parentheses
    cleaned_name = re.sub(r'\s*\(.*?\)', '', track_name)
    # Remove any suffix starting with '-'
    cleaned_name = re.sub(r' - .*$', '', cleaned_name)
    # Return the cleaned name with leading and trailing whitespace removed
    return cleaned_name.strip().lower()



def process_lyrics(df, track_col='track_name', lyrics_col='lyrics'):
    """
    Process the DataFrame to count how many times the cleaned track name appears in the lyrics.
    """
    df['word_count'] = df.apply(lambda row: count_title_in_lyrics(row[track_col], row[lyrics_col]), axis=1)
    return df

def show_artist_and_track(df, index):
    """
    Show the artist_name and track_name for a given index in the DataFrame.
    
    Parameters:
    - df: The DataFrame containing the music data.
    - index: The index of the row you want to inspect.
    
    Returns:
    - A tuple containing (artist_name, track_name) if the index is valid.
    - A message indicating the index is out of bounds if the index is invalid.
    """
    if index in df.index:
        artist_name = df.at[index, 'artist_name']
        track_name = df.at[index, 'track_name']
        return artist_name, track_name
    else:
        return f"Index {index} is out of bounds."


## Gym

In [27]:
gym['track_name_count'] = gym.apply(lambda row: row['lyrics'].lower().count(row['track_name'].lower()), axis=1)
gym_wrong_lyrics = gym[gym['track_name_count'] == 0][['track_name', 'artist_name', 'lyrics', 'track_name_count']]

In [40]:
# Fetch lyrics for wrong gym lyrics
fetch_lyrics_in_batches(api, gym_wrong_lyrics)

Searching for "HUMBLE." by Kendrick Lamar...
Done.
Searching for "Praise The Lord (Da Shine) (feat. Skepta)" by A$AP Rocky...
Done.
Searching for "Superhero (Heroes & Villains) [with Future & Chris Brown]" by Metro Boomin...
Done.
Searching for "DNA." by Kendrick Lamar...
Error fetching lyrics for DNA. by Kendrick Lamar: Request timed out:
HTTPSConnectionPool(host='genius.com', port=443): Read timed out. (read timeout=5)
Searching for "Hypnotize" by The Notorious B.I.G....
Done.
Searching for "Wild Thoughts (feat. Rihanna & Bryson Tiller)" by DJ Khaled...
Done.
Searching for "INDUSTRY BABY (feat. Jack Harlow)" by Lil Nas X...
Done.
Searching for "Ballin' (with Roddy Ricch)" by Mustard...
Done.
Searching for "Boy's a liar Pt. 2" by PinkPantheress...
Done.
Searching for "Wanna Be (with Megan Thee Stallion & Cardi B) - Remix" by GloRilla...
Done.
Searching for "GEEKALEEK (feat. BIA)" by OHGEESY...
Done.
Searching for "FTCU (feat. Travis Scott, Chris Brown & Sexyy Red) - SLEEZEMIX" by Nick

,track_name,artist_name,lyrics,track_name_count,cleaned_track_name
6,HUMBLE.,Kendrick Lamar,1016 ContributorsTranslationsTürkçeEspañolРусс...,0,HUMBLE.
11,Praise The Lord (Da Shine) (feat. Skepta),A$AP Rocky,172 ContributorsRanking Page: 50 Best Albums &...,0,Praise The Lord (Da Shine) (feat. Skepta)
14,Superhero (Heroes & Villains) [with Future & C...,Metro Boomin,1 Contributor2018 haywoodindahood Listening Lo...,0,Superhero (Heroes & Villains) [with Future & C...
19,DNA.,Kendrick Lamar,Error: Request timed out:\nHTTPSConnectionPool...,0,DNA.
22,Hypnotize - 2014 Remaster,The Notorious B.I.G.,228 ContributorsHypnotize Lyrics[Intro: The No...,0,Hypnotize
...,...,...,...,...,...
577,Lay All Your Love On Me,loafers,3 ContributorsThe Beginning. The Separation. L...,0,Lay All Your Love On Me
579,Tattoo - Topic Remix,Loreen,1 ContributorTattoo (Topic Remix) Lyrics[Verse...,0,Tattoo - Topic Remix
581,Turn On The Lights again.. (feat. Future & Fre...,Swedish House Mafia,3 ContributorsTurn On The Lights again.. (Anym...,0,Turn On The Lights again.. (feat. Future & Fre...
592,Rock My Body (with Sash!) - W&W x R3HAB VIP Remix,R3HAB,Lyrics not found,0,Rock My Body (with Sash!) - W&W x R3HAB VIP Remix


In [57]:
from lyricsgenius import Genius

def fetch_lyrics(api, track_name, artist_name):
    """
    Fetches lyrics for a given track name and artist name using the Genius API.
    
    Parameters:
    - api: The Genius API client.
    - track_name: The name of the track.
    - artist_name: The name of the artist.
    
    Returns:
    - lyrics: The lyrics of the song if found, otherwise an appropriate message.
    """
    try:
        # Search for the song using the Genius API
        song = api.search_song(title=track_name, artist=artist_name)
        if song:
            return f"Lyrics for '{track_name}' by {artist_name}:\n\n{song.lyrics}"
        else:
            return f"Lyrics for '{track_name}' by {artist_name} not found."
    except Exception as e:
        return f"An error occurred: {str(e)}"



gym_11 = fetch_lyrics(api, track_name = "Praise the Lord (Da Shine)", artist_name = "A$AP Rocky")
gym_14 = fetch_lyrics(api, track_name = "Superhero (Heroes & Villains)",
                      artist_name = "Metro Boomin, Future & Chris Brown")
gym_19 = fetch_lyrics(api, track_name = "DNA.",
                      artist_name = "Kendrick Lamar")
gym_27 = fetch_lyrics(api, track_name = "Wild Thoughts", artist_name = "DJ Khaled")
gym_45 = fetch_lyrics(api, track_name = "Ballin'", artist_name = "Mustard")
gym_53 = fetch_lyrics(api, track_name = "Wanna Be'", artist_name = "GloRilla")

In [107]:
gym.loc[11, 'lyrics'] = gym_11
gym.loc[19, 'lyrics'] = gym_19
gym.loc[27, 'lyrics'] = gym_27
gym.loc[45, 'lyrics'] = gym_45
gym.loc[53, 'lyrics'] = gym_53
gym.loc[14, 'lyrics'] = gym_14

In [224]:
gym_6 = fetch_lyrics(api, track_name = "HUMBLE.", artist_name = "Kendrick Lamar")
gym_113 = fetch_lyrics(api, track_name = "King Shit", artist_name = "Yo Gotti")
gym_115 = fetch_lyrics(api, track_name = "Tonight Is the Night", artist_name = "Outasight")
gym_226 = fetch_lyrics(api, track_name = "Austin (Boots Stop Workin’)", artist_name = "Dasha")
gym_285 = fetch_lyrics(api, track_name = "SLEEP TONIGHT (THIS IS THE LIFE)", artist_name = "Switch Disco")
gym_338 = fetch_lyrics(api, track_name = "Dance The Night", artist_name = "Dua Lipa")
gym_527 = fetch_lyrics(api, track_name = "Wild World", artist_name = "王若琳 (Joanna Wang)")
gym_533 = fetch_lyrics(api, track_name = "Get Busy", artist_name = "Sean Paul")
gym_544 = 'Yo paseaba sola por las calles Sacando fotos a los monumentos La típica extranjera con un aire extraño \
Que recorre entera toda la ciudad'
gym_547 = fetch_lyrics(api, track_name = "Whatever (Tiësto Remix)", artist_name = "Kygo & Ava Max")
gym_567 = fetch_lyrics(api, track_name = "Like A G6", artist_name = "Timmy Trumpet & POLTERGST")
gym_581 = fetch_lyrics(api, track_name = "Turn On The Lights again..", artist_name = "Fred again.. & Swedish House Mafia")

Searching for "HUMBLE." by Kendrick Lamar...
Done.


In [314]:
gym.loc[6, 'lyrics'] = gym_6
gym.loc[113, 'lyrics'] = gym_113
gym.loc[115, 'lyrics'] = gym_115
gym.loc[226, 'lyrics'] = gym_226
gym.loc[285, 'lyrics'] = gym_285
gym.loc[338, 'lyrics'] = gym_338
gym.loc[527, 'lyrics'] = gym_527
gym.loc[533, 'lyrics'] = gym_533
gym.loc[544, 'lyrics'] = gym_544
gym.loc[547, 'lyrics'] = gym_547
gym.loc[567, 'lyrics'] = gym_567
gym.loc[581, 'lyrics'] = gym_581

## Bar

In [320]:
bar['track_name_count'] = bar.apply(lambda row: row['lyrics'].lower().count(row['track_name'].lower()), axis=1)
new_bar = bar[bar['track_name_count'] == 0][['track_name', 'artist_name', 'lyrics', 'track_name_count']]
new_bar=new_bar[new_bar['lyrics'] != 'Lyrics not found']

new_bar = process_lyrics(new_bar)
new_new_bar=new_bar[new_bar['lyrics'] != 'Lyrics not found']

In [332]:
fetch_lyrics_in_batches(api, new_bar)

Searching for "body" by Loud Luxury...
Done.
Searching for "talk" by Khalid...
Done.
Searching for "cold" by Maroon 5...
Done.
Searching for "attention" by Charlie Puth...
Done.
Searching for "heatstroke" by Calvin Harris...
Done.
Searching for "rockabye" by Clean Bandit...
Done.
Searching for "lost in japan" by Shawn Mendes...
Done.
Searching for "god's plan" by Drake...
Done.
Searching for "circles" by Post Malone...
Done.
Searching for "what lovers do" by Maroon 5...
Done.
Searching for "bad at love" by Halsey...
Done.
Searching for "havana" by Camila Cabello...
Done.
Searching for "the weekend" by SZA...
Done.
Searching for "playinwitme" by KYLE...
Done.
Searching for "moves like jagger" by Maroon 5...
Done.
Searching for "love yourself" by Justin Bieber...
Done.
Searching for "cold water" by Major Lazer...
Done.
Searching for "i took a pill in ibiza" by Mike Posner...
Done.
Searching for "congratulations" by Post Malone...
Done.
Searching for "feels" by Calvin Harris...
Done.
Sear

,track_name,artist_name,lyrics,track_name_count,word_count,cleaned_track_name
1,Body (feat. brando),Loud Luxury,"56 ContributorsBody Lyrics[Verse 1]\nBabe, don...",0,2,body
3,Talk (feat. Disclosure),Khalid,86 ContributorsTranslationsفارسیEspañolTalk Ly...,0,1,talk
5,Cold (feat. Future),Maroon 5,83 ContributorsTranslationsPortuguêsCold Lyric...,0,3,cold
8,Attention - Oliver Heldens Remix,Charlie Puth,140 ContributorsTranslationsEestiالعربيةEspaño...,0,16,attention
9,"Heatstroke (feat. Young Thug, Pharrell William...",Calvin Harris,96 ContributorsTranslationsEnglishHeatstroke L...,0,1,heatstroke
...,...,...,...,...,...,...
474,Theme for Ernie,Bévort 3,8 ContributorsPalm Pilot Lyrics[Skit]\n*Young ...,0,0,theme for ernie
477,Song To An Old Friend,Jakob Bro,31 ContributorsUlysses (Chap. 12 - Cyclops) Ly...,0,0,song to an old friend
478,I Love Music,Jeremy Pelt,"1 ContributorBaby Talk LyricsNARRATOR: Kids, i...",0,0,i love music
480,One Step at a Time,Jan Harbeck Quartet,7 ContributorsThe Making of Paul (Chap. 4: Pau...,0,0,one step at a time


In [466]:
bar_169 = fetch_lyrics(api, track_name = "She Hates Me", artist_name = "Puddle of Mudd")
bar_191 = fetch_lyrics(api, track_name = "Take on Me", artist_name = "a-ha")
bar_255 = fetch_lyrics(api, track_name = "Old Town Road (Remix)", artist_name = "Lil Nas X")
bar_323 = fetch_lyrics(api, track_name = "One Day (Vandaag)", artist_name = "Bakermat")
bar_333 = fetch_lyrics(api, track_name = "Don’t Need Love", artist_name = "220 KID & GRACEY")
bar_368 = fetch_lyrics(api, track_name = "Gotta Let You Go (Original ’95 Club Mix)", artist_name = "Dominica")

Searching for "She Hates Me" by Puddle of Mudd...
Done.


In [473]:
new_bar.loc[169, 'lyrics'] = bar_169
new_bar.loc[191, 'lyrics'] = bar_191
new_bar.loc[255, 'lyrics'] = bar_255
new_bar.loc[323, 'lyrics'] = bar_323
bar.loc[333, 'lyrics'] = bar_333
new_bar.loc[368, 'lyrics'] = bar_368

## Bedroom

In [483]:
bedroom['track_name_count'] = bar.apply(lambda row: row['lyrics'].lower().count(row['track_name'].lower()), axis=1)
new_bed = bedroom[bedroom['track_name_count'] == 0][['track_name', 'artist_name', 'lyrics', 'track_name_count']]
new_bed=new_bed[new_bed['lyrics'] != 'Lyrics not found']

new_bed = process_lyrics(new_bed)

new_new_bed=new_bed[new_bed['lyrics'] != 'Lyrics not found']

In [485]:
fetch_lyrics_in_batches(api, new_bed)

Searching for "blow [feat. kilo ali]" by Sosa UK...
Done.
Searching for "waiting for tonight" by FISHER...
Done.
Searching for "she's in the club" by MK...
Done.
Searching for "girl$" by Dom Dolla...
Done.
Searching for "all that you need" by Gorgon City...
Done.
Searching for "wild" by James Hype...
Done.
Searching for "lobster telephone" by Peggy Gou...
Done.
Searching for "delirious" by Leftwing : Kody...
Done.
Searching for "under my feet" by Selena Faider...
Done.
Searching for "incanto" by Ryan Arnold...
Done.
Searching for "lemonade" by Deeper Purpose...
Done.
Searching for "to l.a" by Tita Lau...
Done.
Searching for "in your arms" by Jess Bays...
Done.
Searching for "the weekend" by Anti Up...
Done.
Searching for "vibrate" by James Hype...
Done.
Searching for "anthem" by Diplo...
Done.
Searching for "one by one" by Deeper Purpose...
Done.
Searching for "desire" by Chris Stussy...
Done.
Searching for "kissin' on my tattoos" by August Alsina...
Done.
Searching for "2012" by Chris

,track_name,artist_name,lyrics,track_name_count,word_count,cleaned_track_name
1,BLOW (WHITE GIRL IN TOWN) [feat. Kilo Ali],Sosa UK,3 ContributorsBLOW (WHITE GIRL IN TOWN) Lyrics...,0.0,0,blow [feat. kilo ali]
3,Waiting For Tonight,FISHER,5 ContributorsWaiting for Tonight Lyrics[Choru...,0.0,14,waiting for tonight
5,She's In The Club (feat. Asal),MK,5 ContributorsShe’s In the Club Lyrics[Verse 1...,0.0,8,she's in the club
8,girl$,Dom Dolla,"5 Contributors​girl$ Lyrics[Intro]\nWoah\nHey,...",0.0,1,girl$
9,All That You Need (feat. Caroline Byrne),Gorgon City,3 ContributorsAll That You Need Lyrics[Intro]\...,0.0,3,all that you need
...,...,...,...,...,...,...
476,Lo Que Soy,Wax Motif,"3 ContributorsLo Que Soy Lyrics[Letra de ""Lo Q...",0.0,16,lo que soy
477,Monaco,Blank Sense,9 ContributorsForeign Kix Lyrics[Chorus: Erick...,0.0,0,monaco
479,Count On My Love,The Blessed Madonna,4 ContributorsCount On My Love Lyrics[Verse]\n...,0.0,7,count on my love
482,Feel,LP Giobbi,3 ContributorsFeel LyricsOh I need this feel f...,0.0,16,feel


In [612]:
bed_168 = fetch_lyrics(api, track_name = "No Bullshit", artist_name = "Chris Brown")
bed_240 = fetch_lyrics(api, track_name = "Take Me Away", artist_name = "DJ S.K.T")
bed_455 = fetch_lyrics(api, track_name = "Brighter Days (Marco Lys Remix)", artist_name = "Cajmere")
bed_458 = fetch_lyrics(api, track_name = "Can’t Play Myself (A Tribute To Amy)", artist_name = "Skepta")

Searching for "No Bullshit" by Chris Brown...
Done.


In [618]:
new_bed.loc[168, 'lyrics'] = bed_168
new_bed.loc[240, 'lyrics'] = bed_240
new_bed.loc[455, 'lyrics'] = bed_455
new_bed.loc[458, 'lyrics'] = bed_458

In [619]:
bedroom.loc[234, 'lyrics'] = 'Lyrics not found'

In [624]:
bedroom.loc[new_bed.index, 'lyrics'] = new_bed['lyrics']

In [700]:
bed_6 = fetch_lyrics(api, track_name = "Touch Down", artist_name = "Mella Dee & Storm Mollison")
bed_7 = fetch_lyrics(api, track_name = "Sweet Devotion", artist_name = "TSHA & Caroline Byrne")
bed_64 = fetch_lyrics(api, track_name = "That Sound", artist_name = "Deeper Purpose & GUZ")
bed_139 = fetch_lyrics(api, track_name = "The Take", artist_name = "Tory Lanez")
bed_150 = fetch_lyrics(api, track_name = "It Won’t Stop (Remix)", artist_name = "Sevyn Streeter")
bed_163 = fetch_lyrics(api, track_name = "How You Want It?", artist_name = "Teyana Taylor")
bed_164 = fetch_lyrics(api, track_name = "Earned It", artist_name = "The Weeknd")
bed_231 = fetch_lyrics(api, track_name = "I Went Too Far", artist_name = "AURORA")
bed_246 = fetch_lyrics(api, track_name = "Won't Look Back", artist_name = "Duke Dumont")
bed_252 = fetch_lyrics(api, track_name = "Too Much to Handle (XYconstant Remix)",
                       artist_name = "Great Good Fine OK")
bed_262 = fetch_lyrics(api, track_name = "Feel My Needs", artist_name = "Weiss")
bed_361 = fetch_lyrics(api, track_name = "All My Friends", artist_name = "Snakehips")
bed_530 = fetch_lyrics(api, track_name = "Brighter", artist_name = "Kaskade & Emmit Fenn")
bed_533 = fetch_lyrics(api, track_name = "Time", artist_name = "Carlita (EDM)")
bed_542 = fetch_lyrics(api, track_name = "The Chase", artist_name = "Emmit Fenn")
bed_558 = fetch_lyrics(api, track_name = "It’s Ok It’s Alright", artist_name = "Emmit Fenn")

Searching for "Touch Down" by Mella Dee & Storm Mollison...
Done.


In [717]:
bedroom.loc[6, 'lyrics'] = bed_6
bedroom.loc[7, 'lyrics'] = bed_7
bedroom.loc[139, 'lyrics'] = bed_139
bedroom.loc[150, 'lyrics'] = bed_150
bedroom.loc[163, 'lyrics'] = bed_163
bedroom.loc[164, 'lyrics'] = bed_164
bedroom.loc[231, 'lyrics'] = bed_231
bedroom.loc[246, 'lyrics'] = bed_246
bedroom.loc[252, 'lyrics'] = bed_252
bedroom.loc[262, 'lyrics'] = bed_262
bedroom.loc[361, 'lyrics'] = bed_361
bedroom.loc[530, 'lyrics'] = bed_530
bedroom.loc[533, 'lyrics'] = bed_533
bedroom.loc[542, 'lyrics'] = bed_542
bedroom.loc[558, 'lyrics'] = bed_558

## Mall

In [722]:
mall['track_name_count'] = mall.apply(lambda row: row['lyrics'].lower().count(row['track_name'].lower()), axis=1)
nextt = mall[mall['track_name_count'] == 0][['track_name', 'artist_name', 'lyrics', 'track_name_count']]
nextt=nextt[nextt['lyrics'] != 'Lyrics not found']

nextt = process_lyrics(nextt)

In [723]:
nextt

,track_name,artist_name,lyrics,track_name_count,word_count
2,Dance The Night - From Barbie The Album,Dua Lipa,2 Contributors2024 Nominees LyricsGeneral Fiel...,0,1
14,we can't be friends (wait for your love),Ariana Grande,119 ContributorsTranslationsالعربيةБългарскиDa...,0,3
22,You Make My Heart Go,Sad Puppy,1 ContributorTime Travel LyricsTIME TRAVEL\nSH...,0,0
26,Rush (feat. PinkPantheress & Hyunjin of Stray ...,Troye Sivan,16 ContributorsAugust 2023 Singles Release Cal...,0,3
29,Come On Over,the dreamchild,4 ContributorsThe Year of the Underdogz Lyrics...,0,0
...,...,...,...,...,...
197,Here Comes The Sun - Remastered 2009,The Beatles,3 ContributorsDizaster vs. Soul Khan Lyrics[Ro...,0,0
207,California Dreamin' - Single Version,The Mamas & The Papas,4 ContributorsThe 500 Greatest Songs of All Ti...,0,1
211,Beast Of Burden - Remastered 1994,The Rolling Stones,2 ContributorsRollback LyricsThe elections of ...,0,0
213,Bloom - Bonus Track,The Paper Kites,70 ContributorsSeptember 2023 Album Release Ca...,0,0


In [724]:
fetch_lyrics_in_batches(api, nextt)

Searching for "dance the night" by Dua Lipa...
Done.
Searching for "we can't be friends" by Ariana Grande...
Done.
Searching for "you make my heart go" by Sad Puppy...
Done.
Searching for "rush" by Troye Sivan...
Done.
Searching for "come on over" by the dreamchild...
Done.
Searching for "she's not me" by Sad Puppy...
Done.
Searching for "all i want" by Sad Puppy...
Done.
Searching for "rockstars" by Honest Regrets...
Done.
Searching for "die for you" by The Weeknd...
Done.
Searching for "you're not alone" by Sunday Shirt...
Done.
Searching for "i'm good to go" by Balcony Beats...
Done.
Searching for "left right" by Coppermines...
Done.
Searching for "desire" by Calvin Harris...
Done.
Searching for "sweet torture" by Big Z...
Done.
Searching for "alien man" by Sad Puppy...
Done.
Searching for "just friends" by Balcony Beats...
Done.
Searching for "deep down" by Alok...
Done.
Searching for "so easy to love" by Sad Puppy...
Done.
Searching for "good girls" by Balcony Beats...
Done.
Searc

,track_name,artist_name,lyrics,track_name_count,word_count,cleaned_track_name
2,Dance The Night - From Barbie The Album,Dua Lipa,63 ContributorsTranslationsPortuguêsEspañolFra...,0,1,dance the night
14,we can't be friends (wait for your love),Ariana Grande,119 ContributorsTranslationsالعربيةБългарскиDa...,0,3,we can't be friends
22,You Make My Heart Go,Sad Puppy,1 ContributorTime Travel LyricsTIME TRAVEL\nSH...,0,0,you make my heart go
26,Rush (feat. PinkPantheress & Hyunjin of Stray ...,Troye Sivan,63 ContributorsTranslationsEspañolPortuguêsFra...,0,3,rush
29,Come On Over,the dreamchild,4 ContributorsThe Year of the Underdogz Lyrics...,0,0,come on over
...,...,...,...,...,...,...
197,Here Comes The Sun - Remastered 2009,The Beatles,99 ContributorsTranslationsEspañolPortuguêsHer...,0,0,here comes the sun
207,California Dreamin' - Single Version,The Mamas & The Papas,79 ContributorsTranslationsTürkçeCymraegEspaño...,0,1,california dreamin'
211,Beast Of Burden - Remastered 1994,The Rolling Stones,33 ContributorsBeast of Burden Lyrics[Intro]\n...,0,0,beast of burden
213,Bloom - Bonus Track,The Paper Kites,28 ContributorsBloom Lyrics[Verse 1]\nIn the m...,0,0,bloom


In [793]:
mall.loc[nextt.index, 'lyrics'] = nextt['lyrics']

## Library

In [796]:
library['track_name_count'] = library.apply(lambda row: row['lyrics'].lower().count(row['track_name'].lower()), axis=1)
nextt = library[library['track_name_count'] == 0][['track_name', 'artist_name', 'lyrics', 'track_name_count']]
nextt=nextt[nextt['lyrics'] != 'Lyrics not found']

nextt = process_lyrics(nextt)

In [799]:
library['lyrics'].loc[2]

'Lyrics not found'

In [833]:
lib_126 = fetch_lyrics(api, track_name = "Bailar", artist_name = "Deorro")

Searching for "Bailar" by Deorro...
Done.


In [834]:
lib_137 = fetch_lyrics(api, track_name = "Whatever (Tiësto Remix)", artist_name = "Kygo & Ava Max")

Searching for "Whatever (Tiësto Remix)" by Kygo & Ava Max...
Done.


In [835]:
lib_150 = fetch_lyrics(api, track_name = "Without You", artist_name = "Avicii")

Searching for "Without You" by Avicii...
Done.


In [836]:
lib_151 = fetch_lyrics(api, track_name = "Alors on danse", artist_name = "Stromae")

Searching for "Alors on danse" by Stromae...
Done.


In [837]:
lib_157 = fetch_lyrics(api, track_name = "LET'S GET FKD UP", artist_name = "Alok, Mondello’G & Ceres")

Searching for "LET'S GET FKD UP" by Alok, Mondello’G & Ceres...
Done.


In [838]:
lib_293 = fetch_lyrics(api, track_name = "The Way", artist_name = "Kehlani")

Searching for "The Way" by Kehlani...
Done.


In [839]:
lib_299 = fetch_lyrics(api, track_name = "Ring", artist_name = "Cardi B")

Searching for "Ring" by Cardi B...
Done.


In [840]:
lib_319 = fetch_lyrics(api, track_name = "Love All", artist_name = "Drake")

Searching for "Love All" by Drake...
Done.


In [841]:
lib_322 = fetch_lyrics(api, track_name = "WAIT FOR U", artist_name = "Future")

Searching for "WAIT FOR U" by Future...
Done.


In [842]:
lib_327 = fetch_lyrics(api, track_name = "White Tee", artist_name = "Summer Walker & NO1-NOAH")

Searching for "White Tee" by Summer Walker & NO1-NOAH...
Done.


In [849]:
lib_335 = fetch_lyrics(api, track_name = "Playing Games (Extended Version)", artist_name = "Summer Walker & Bryson Tiller")

Searching for "Playing Games (Extended Version)" by Summer Walker & Bryson Tiller...
Done.


In [845]:
lib_349 = fetch_lyrics(api, track_name = "Drama", artist_name = "Roy Woods")

Searching for "Drama" by Roy Woods...
Done.


In [846]:
lib_352 = fetch_lyrics(api, track_name = "love nwantiti (ah ah ah)", artist_name = "CKay")

Searching for "love nwantiti (ah ah ah)" by CKay...
Done.


In [847]:
lib_360 = fetch_lyrics(api, track_name = "Say My Name", artist_name = "Destiny’s Child")

Searching for "Say My Name" by Destiny’s Child...
Done.


In [848]:
lib_362 = fetch_lyrics(api, track_name = "Girls Need Love (Remix)", artist_name = "Summer Walker & Drake")

Searching for "Girls Need Love (Remix)" by Summer Walker & Drake...
Done.


In [850]:
show_artist_and_track(nextt, 151)

('Stromae', 'Alors on danse - Radio Edit')

In [851]:
nextt.loc[126, 'lyrics'] = lib_126
nextt.loc[137, 'lyrics'] = lib_137
nextt.loc[150, 'lyrics'] = lib_150
nextt.loc[151, 'lyrics'] = lib_151
nextt.loc[157, 'lyrics'] = lib_157
nextt.loc[293, 'lyrics'] = lib_293
nextt.loc[299, 'lyrics'] = lib_299
nextt.loc[319, 'lyrics'] = lib_319
nextt.loc[322, 'lyrics'] = lib_322
nextt.loc[327, 'lyrics'] = lib_327
nextt.loc[335, 'lyrics'] = lib_335
nextt.loc[349, 'lyrics'] = lib_349
nextt.loc[352, 'lyrics'] = lib_352
nextt.loc[360, 'lyrics'] = lib_360
nextt.loc[362, 'lyrics'] = lib_362

In [853]:
library.loc[nextt.index, 'lyrics'] = nextt['lyrics']

## Office

In [855]:
office['track_name_count'] = office.apply(lambda row: row['lyrics'].lower().count(row['track_name'].lower()), axis=1)
nextt = office[office['track_name_count'] == 0][['track_name', 'artist_name', 'lyrics', 'track_name_count']]
nextt=nextt[nextt['lyrics'] != 'Lyrics not found']

nextt = process_lyrics(nextt)

In [857]:
fetch_lyrics_in_batches(api, nextt)

Searching for "house of deer" by VIMES...
Done.
Searching for "millions of time" by Mountains...
Done.
Searching for "west" by Hammock...
Done.
Searching for "stasis" by Solar Bears...
Done.
Searching for "k&f thema" by Apparat...
Done.
Searching for "la femme d'argent" by Air...
Done.
Searching for "down under the manhattan bridge overpass" by Mountains...
Done.
Searching for "stars" by Brian Eno...
Specified song does not contain lyrics. Rejecting.
Searching for "structures from silence" by Steve Roach...
Specified song does not contain lyrics. Rejecting.
Searching for "a feeling of being closer" by Antonymes...
Done.
Searching for "unusual thursday" by Darshan Ambient...
No results found for: 'unusual thursday Darshan Ambient'
Searching for "leaving the tetons" by Mark Preston...
Done.
Searching for "same rules apply" by Clint Mansell...
Done.
Searching for "the mysterious fish named kun" by Aglaia...
Done.
Searching for "the tribe of the flying monkeys" by Aglaia...
Done.
Searching

,track_name,artist_name,lyrics,track_name_count,word_count,cleaned_track_name
0,House of Deer,VIMES,11 ContributorsFinnegan’s Wake (Chap. 2.3) Lyr...,0,0,house of deer
1,Millions of Time,Mountains,7 ContributorsCircus of Heaven Lyrics[Verse 1]...,0,0,millions of time
7,West,Hammock,3 ContributorsMoby-Dick (Chap. 126: The Life-B...,0,0,west
11,Stasis,Solar Bears,1 ContributorI Played Everything LyricsI’ve pl...,0,0,stasis
16,K&F Thema,Apparat,2 ContributorsZelda LyricsCutting right up whe...,0,0,k&f thema
...,...,...,...,...,...,...
700,Perfect Duet (Ed Sheeran & Beyoncé),Ed Sheeran,81 ContributorsPerfect Duet Lyrics[Verse 1: Ed...,0,1,perfect duet
701,True - Single Edit,Spandau Ballet,50 ContributorsTrue Lyrics[Intro]\nHuh huh uh-...,0,1,true
722,(What A) Wonderful World,Sam Cooke,30 ContributorsTranslationsDeutschWonderful Wo...,0,3,wonderful world
727,I Don't Feel Like Dancin' - Radio Edit,Scissor Sisters,29 ContributorsI Don’t Feel Like Dancin’ Lyric...,0,6,i don't feel like dancin'


In [906]:
off_100 = fetch_lyrics(api, track_name = "One Dance", artist_name = "Drake")
off_109 = fetch_lyrics(api, track_name = "Hey Ya!", artist_name = "OutKast")
off_203 = fetch_lyrics(api, track_name = "What Goes Around... Comes Around (Interlude)", artist_name = "Justin Timberlake")
off_335 = fetch_lyrics(api, track_name = "Fashion Killa", artist_name = "Ayra Starr")

Searching for "One Dance" by Drake...
Done.


In [915]:
nextt.loc[100, 'lyrics'] = off_100
nextt.loc[109, 'lyrics'] = off_109
nextt.loc[203, 'lyrics'] = off_203
nextt.loc[335, 'lyrics'] = off_335

In [917]:
office.loc[nextt.index, 'lyrics'] = nextt['lyrics']

## Restaurant

In [919]:
restaurant['track_name_count'] = restaurant.apply(lambda row: row['lyrics'].lower().count(row['track_name'].lower()), axis=1)
nextt = restaurant[restaurant['track_name_count'] == 0][['track_name', 'artist_name', 'lyrics', 'track_name_count']]
nextt=nextt[nextt['lyrics'] != 'Lyrics not found']

nextt = process_lyrics(nextt)

In [921]:
fetch_lyrics_in_batches(api, nextt)

Searching for "lovely day" by Bill Withers...
Done.
Searching for "lavender haze" by Taylor Swift...
Done.
Searching for "some things i'll never know" by Teddy Swims...
Done.
Searching for "watermelon sugar" by Emei...
Done.
Searching for "lift me up" by Rihanna...
Done.
Searching for "july" by Noah Cyrus...
Done.
Searching for "this must be the place" by Talking Heads...
Done.
Searching for "one of these nights" by Eagles...
Done.
Searching for "and i love her" by The Beatles...
Done.
Searching for "what you won't do for love" by Bobby Caldwell...
Done.
Searching for "a thousand miles" by Vanessa Carlton...
Done.
Searching for "bloom" by The Paper Kites...
Done.
Searching for "always on my mind" by Nané...
Done.
Searching for "sun & moon" by BALTHVS...
Done.
Searching for "nantucket island" by Willie Wright...
Done.
Searching for "mambo sun" by T. Rex...
Done.
Searching for "people never change" by Peter Cat Recording Co....
Done.
Searching for "feed the fire" by SG Lewis...
Done.
Sea

,track_name,artist_name,lyrics,track_name_count,word_count,cleaned_track_name
0,Lovely Day - Studio Rio Version,Bill Withers,46 ContributorsLovely Day Lyrics[Verse 1]\nWhe...,0,0,lovely day
4,Lavender Haze - Acoustic Version,Taylor Swift,126 ContributorsTranslationsEspañolFrançaisPor...,0,7,lavender haze
6,Some Things I'll Never Know (feat. Maren Morris),Teddy Swims,5 ContributorsTranslationsУкраїнськаSome Thing...,0,0,some things i'll never know
12,Watermelon Sugar - Spotify Singles,Emei,4 ContributorsWatermelon Sugar (Spotify Single...,0,22,watermelon sugar
36,Lift Me Up - From Black Panther: Wakanda Forev...,Rihanna,109 ContributorsTranslationsPortuguêsTürkçeEsp...,0,1,lift me up
...,...,...,...,...,...,...
575,People Everyday - Metamorphosis Mix,Arrested Development,1 ContributorPeople Everyday (unplugged) Lyric...,0,0,people everyday
576,Thank You (Falettinme Be Mice Elf Agin) - Sing...,Sly & The Family Stone,2 ContributorsThanx 4 the Funk Lyrics[Intro]\n...,0,0,thank you
577,Feels Good,Tony! Toni! Toné!,21 ContributorsFeels Good Lyrics[Intro]\nPlay ...,0,0,feels good
578,Before I Let Go - Homecoming Live Bonus Track,Beyoncé,63 ContributorsTranslationsBeyoncé - Before I ...,0,0,before i let go


In [1056]:
restaurant.loc[181, 'artist_name'] = "The Marías"
restaurant.loc[207, 'artist_name'] = "Anaïs Mitchell"
restaurant.loc[215, 'artist_name'] = "Tré Burt"
restaurant.loc[443, 'track_name'] = "Boy’s a liar Pt. 2 (Sped Up)"

In [1058]:
rest_518 = fetch_lyrics(api, track_name = "Don't Stop 'Til You Get Enough", artist_name = "Michael Jackson")
rest_537 = fetch_lyrics(api, track_name = "Back to Life (However Do You Want Me)", artist_name = "Soul II Soul")
rest_543 = fetch_lyrics(api, track_name = "Int’l Players Anthem (I Choose You)", artist_name = "UGK")
rest_547 = fetch_lyrics(api, track_name = "One More Chance / Stay With Me (Remix))", artist_name = "The Notorious B.I.G.")
rest_576 = fetch_lyrics(api, track_name = "Thank You (Falettinme Be Mice Elf Agin)", artist_name = "Sly and the Family Stone")
rest_579 = fetch_lyrics(api, track_name = "All Night Long", artist_name = "Mary Jane Girls")
rest_181 = fetch_lyrics(api, track_name = "Run Your Mouth", artist_name = "The Marías")
rest_207 = fetch_lyrics(api, track_name = "Bright Star", artist_name = "Anaïs Mitchell")
rest_215 = fetch_lyrics(api, track_name = "Sweet Misery", artist_name = "Tré Burt")
rest_363 = fetch_lyrics(api, track_name = "Shape of You", artist_name = "Ed Sheeran")
rest_388 = fetch_lyrics(api, track_name = "Viva La Vida", artist_name = "Coldplay")
rest_392 = fetch_lyrics(api, track_name = "The Night We Met (Remix)", artist_name = "Lord Huron")
rest_394 = fetch_lyrics(api, track_name = "Young and Beautiful", artist_name = "Lana Del Rey")
rest_398 = fetch_lyrics(api, track_name = "Numb", artist_name = "Linkin Park")
rest_399 = fetch_lyrics(api, track_name = "Te Amo", artist_name = "Rihanna")
rest_400 = fetch_lyrics(api, track_name = "Everytime We Touch", artist_name = "Cascada")
rest_403 = fetch_lyrics(api, track_name = "I’m Good (Blue)", artist_name = "David Guetta & Bebe Rexha")
rest_404 = fetch_lyrics(api, track_name = "Cupid’s Chokehold / Breakfast in America", artist_name = "Gym Class Heroes")
rest_406 = fetch_lyrics(api, track_name = "Lift Me Up", artist_name = "Rihanna")
rest_407 = fetch_lyrics(api, track_name = "Dancing on My Own", artist_name = "Calum Scott")
rest_409 = fetch_lyrics(api, track_name = "Smack That", artist_name = "Akon")
rest_410 = fetch_lyrics(api, track_name = "As It Was", artist_name = "Harry Styles")
rest_411 = fetch_lyrics(api, track_name = "Space Song (Slowed and Reverb)", artist_name = "Beach House")
rest_412 = fetch_lyrics(api, track_name = "Don’t Let Me Down", artist_name = "The Chainsmokers")
rest_414 = fetch_lyrics(api, track_name = "Mary on a Cross", artist_name = "Ghost")
rest_418 = fetch_lyrics(api, track_name = "Titanium", artist_name = "David Guetta")
rest_419 = fetch_lyrics(api, track_name = "Disturbia", artist_name = "Rihanna")
rest_420 = fetch_lyrics(api, track_name = "Crazy", artist_name = "Gnarls Barkley")
rest_422 = fetch_lyrics(api, track_name = "Big Jet Plane", artist_name = "Angus & Julia Stone")
rest_423 = fetch_lyrics(api, track_name = "Flower", artist_name = "Miley Cyrus")
rest_425 = fetch_lyrics(api, track_name = "Apologize", artist_name = "Timbaland")
rest_426 = fetch_lyrics(api, track_name = "Shameless", artist_name = "Camila Cabello")
rest_430 = fetch_lyrics(api, track_name = "Big Jet Plane", artist_name = "Angus & Julia Stone")
rest_431 = fetch_lyrics(api, track_name = "The Night We Met (Remix)", artist_name = "Lord Huron")
rest_432 = fetch_lyrics(api, track_name = "All of Me", artist_name = "John Legend")
rest_433 = fetch_lyrics(api, track_name = "Closer", artist_name = "The Chainsmokers")
rest_434 = fetch_lyrics(api, track_name = "Lost", artist_name = "Linkin Park")
rest_436 = fetch_lyrics(api, track_name = "Play Hard", artist_name = "David Guetta")
rest_437 = fetch_lyrics(api, track_name = "Where Have You Been", artist_name = "Rihanna")
rest_438 = fetch_lyrics(api, track_name = "Unfaithful", artist_name = "Rihanna")
rest_439 = fetch_lyrics(api, track_name = "Young and Beautiful", artist_name = "Lana Del Rey")
rest_440 = fetch_lyrics(api, track_name = "Rush", artist_name = "Ayra Starr")
rest_441 = fetch_lyrics(api, track_name = "Say Yes To Heaven", artist_name = "Lana Del Rey")
rest_443 = fetch_lyrics(api, track_name = "Boy’s a liar Pt. 2", artist_name = "PinkPantheress & Ice Spice")
rest_445 = fetch_lyrics(api, track_name = "Lose You To Love Me", artist_name = "Selena Gomez")
rest_446 = fetch_lyrics(api, track_name = "Dancing With Your Ghost", artist_name = "Sasha Alex Sloan")
rest_447 = fetch_lyrics(api, track_name = "The Grants", artist_name = "Lana Del Rey")
rest_448 = fetch_lyrics(api, track_name = "Hot N Cold", artist_name = "Katy Perry")
rest_449 = fetch_lyrics(api, track_name = "All Of The Girls You Loved Before", artist_name = "Taylor Swift")
rest_450 = fetch_lyrics(api, track_name = "Te Amo", artist_name = "Rihanna")
rest_451 = fetch_lyrics(api, track_name = "Just Dance", artist_name = "Lady Gaga")
rest_452 = fetch_lyrics(api, track_name = "Give It To Me", artist_name = "Timbaland")
rest_453 = fetch_lyrics(api, track_name = "Creepin'", artist_name = "Metro Boomin, The Weeknd & 21 Savage")
rest_454 = fetch_lyrics(api, track_name = "ceilings", artist_name = "Lizzy McAlpine")
rest_455 = fetch_lyrics(api, track_name = "Say Yes To Heaven", artist_name = "Lana Del Rey")
rest_456 = fetch_lyrics(api, track_name = "Te Amo", artist_name = "Rihanna")
rest_457 = fetch_lyrics(api, track_name = "Trance", artist_name = "Metro Boomin, Travis Scott & Young Thug")
rest_458 = fetch_lyrics(api, track_name = "Kill Bill", artist_name = "SZA")
rest_459 = fetch_lyrics(api, track_name = "Bloody Mary", artist_name = "Lady Gaga")
rest_460 = fetch_lyrics(api, track_name = "It's Called: Freefall", artist_name = "Rainbow Kitten Surprise")
rest_461 = fetch_lyrics(api, track_name = "Viva La Vida", artist_name = "Coldplay")
rest_462 = fetch_lyrics(api, track_name = "If We Ever Broke Up", artist_name = "Mae Stephens")
rest_463 = fetch_lyrics(api, track_name = "Pretty Girls Walk", artist_name = "Big Boss Vette")
rest_464 = fetch_lyrics(api, track_name = "Mad Love", artist_name = "Mabel")
rest_465 = fetch_lyrics(api, track_name = "Cupid (Twin Version)", artist_name = "FIFTY FIFTY (피프티피프티)")
rest_466 = fetch_lyrics(api, track_name = "Whenever, Wherever", artist_name = "Shakira")
rest_467 = fetch_lyrics(api, track_name = "Die For You", artist_name = "The Weeknd")
rest_468 = fetch_lyrics(api, track_name = "Hundred Miles (Extended Mix)", artist_name = "Yall & Gabriela Richardson")
rest_469 = fetch_lyrics(api, track_name = "Radio", artist_name = "Lana Del Rey")

Searching for "Don't Stop 'Til You Get Enough" by Michael Jackson...
Done.


In [1134]:
show_artist_and_track(nextt, 404)

('Way 2 Fast', "Cupid's Chokehold / Breakfast in America - sped up")

In [1135]:
nextt.loc[469, 'lyrics'] = rest_469
nextt.loc[468, 'lyrics'] = rest_468
nextt.loc[467, 'lyrics'] = rest_467
nextt.loc[466, 'lyrics'] = rest_466
nextt.loc[465, 'lyrics'] = rest_465
nextt.loc[464, 'lyrics'] = rest_464
nextt.loc[463, 'lyrics'] = rest_463
nextt.loc[462, 'lyrics'] = rest_462
nextt.loc[461, 'lyrics'] = rest_461
nextt.loc[460, 'lyrics'] = rest_460
nextt.loc[459, 'lyrics'] = rest_459
nextt.loc[458, 'lyrics'] = rest_458
nextt.loc[457, 'lyrics'] = rest_457
nextt.loc[456, 'lyrics'] = rest_456
nextt.loc[455, 'lyrics'] = rest_455
nextt.loc[454, 'lyrics'] = rest_454
nextt.loc[453, 'lyrics'] = rest_453
nextt.loc[452, 'lyrics'] = rest_452
nextt.loc[451, 'lyrics'] = rest_451
nextt.loc[450, 'lyrics'] = rest_450
nextt.loc[449, 'lyrics'] = rest_449
nextt.loc[448, 'lyrics'] = rest_448
nextt.loc[447, 'lyrics'] = rest_447
nextt.loc[446, 'lyrics'] = rest_446
nextt.loc[445, 'lyrics'] = rest_445
nextt.loc[443, 'lyrics'] = rest_443
nextt.loc[441, 'lyrics'] = rest_441
nextt.loc[440, 'lyrics'] = rest_440
nextt.loc[439, 'lyrics'] = rest_439
nextt.loc[438, 'lyrics'] = rest_438
nextt.loc[437, 'lyrics'] = rest_437
nextt.loc[436, 'lyrics'] = rest_436
nextt.loc[434, 'lyrics'] = rest_434
nextt.loc[433, 'lyrics'] = rest_433
nextt.loc[432, 'lyrics'] = rest_432
nextt.loc[431, 'lyrics'] = rest_431
nextt.loc[430, 'lyrics'] = rest_430
nextt.loc[426, 'lyrics'] = rest_426
nextt.loc[425, 'lyrics'] = rest_425
nextt.loc[423, 'lyrics'] = rest_423
nextt.loc[422, 'lyrics'] = rest_422
nextt.loc[420, 'lyrics'] = rest_420
nextt.loc[419, 'lyrics'] = rest_419
nextt.loc[418, 'lyrics'] = rest_418
nextt.loc[414, 'lyrics'] = rest_414
nextt.loc[412, 'lyrics'] = rest_412
nextt.loc[411, 'lyrics'] = rest_411
nextt.loc[410, 'lyrics'] = rest_410
nextt.loc[409, 'lyrics'] = rest_409
nextt.loc[407, 'lyrics'] = rest_407
nextt.loc[406, 'lyrics'] = rest_406
nextt.loc[404, 'lyrics'] = rest_404
nextt.loc[403, 'lyrics'] = rest_403
nextt.loc[400, 'lyrics'] = rest_400
nextt.loc[399, 'lyrics'] = rest_399
nextt.loc[398, 'lyrics'] = rest_398
nextt.loc[394, 'lyrics'] = rest_394
nextt.loc[392, 'lyrics'] = rest_392
nextt.loc[388, 'lyrics'] = rest_388
nextt.loc[363, 'lyrics'] = rest_363
nextt.loc[215, 'lyrics'] = rest_215
nextt.loc[207, 'lyrics'] = rest_207
nextt.loc[181, 'lyrics'] = rest_181
nextt.loc[579, 'lyrics'] = rest_579
nextt.loc[576, 'lyrics'] = rest_576
nextt.loc[547, 'lyrics'] = rest_547
nextt.loc[543, 'lyrics'] = rest_543
nextt.loc[537, 'lyrics'] = rest_537
nextt.loc[518, 'lyrics'] = rest_518

In [1136]:
restaurant.loc[nextt.index, 'lyrics'] = nextt['lyrics']

## TrainBus

In [1143]:
trainBus['track_name_count'] = trainBus.apply(lambda row: row['lyrics'].lower().count(row['track_name'].lower()), axis=1)
nextt = trainBus[trainBus['track_name_count'] == 0][['track_name', 'artist_name', 'lyrics', 'track_name_count']]
nextt=nextt[nextt['lyrics'] != 'Lyrics not found']

nextt = process_lyrics(nextt)

In [1145]:
fetch_lyrics_in_batches(api, nextt)

Searching for "no scrubs" by TLC...
Done.
Searching for "hypnotize" by The Notorious B.I.G....
Done.
Searching for "what's luv?" by Fat Joe...
Done.
Searching for "crazy in love" by Beyoncé...
Done.
Searching for "jenny from the block" by Jennifer Lopez...
Done.
Searching for "yeah!" by USHER...
Done.
Searching for "99 problems" by JAY-Z...
Error fetching lyrics for 99 problems by JAY-Z: Request timed out:
HTTPSConnectionPool(host='genius.com', port=443): Read timed out. (read timeout=5)
Searching for "la tortura" by Shakira...
Done.
Searching for "my name is" by Eminem...
Error fetching lyrics for my name is by Eminem: Request timed out:
HTTPSConnectionPool(host='genius.com', port=443): Read timed out. (read timeout=5)
Searching for "independent women, pt. 1" by Destiny's Child...
Done.
Searching for "shake ya ass" by Mystikal...
Done.
Searching for "bump n' grind" by R. Kelly...
Done.
Searching for "keep on, keepin' on" by MC Lyte...
Done.
Searching for "low" by Flo Rida...
Done.
Sea

,track_name,artist_name,lyrics,track_name_count,word_count,cleaned_track_name
0,No Scrubs,TLC,14 ContributorsTranslationsPortuguêsNo Scrubs ...,0,0,no scrubs
5,Hypnotize - 2007 Remaster,The Notorious B.I.G.,228 ContributorsHypnotize Lyrics[Intro: The No...,0,2,hypnotize
7,What's Luv? (feat. Ja-Rule & Ashanti),Fat Joe,54 ContributorsTranslationsFrançaisWhat’s Luv?...,0,0,what's luv?
9,Crazy In Love (feat. Jay-Z),Beyoncé,122 ContributorsCrazy in Love Lyrics[Intro: JA...,0,1,crazy in love
13,Jenny from the Block (feat. Jadakiss & Styles ...,Jennifer Lopez,39 ContributorsJenny from the Block Lyrics[Int...,0,11,jenny from the block
...,...,...,...,...,...,...
432,GAMMAL I GEMET,Hooja,1 ContributorVem Som Helst Lyrics[Intro]\nHoho...,0,0,gammal i gemet
435,Dansa i neon (N!NE Remix),Lena Philipsson,4 ContributorsDansa I Neon LyricsMitt i en sto...,0,0,dansa i neon
437,Vi ska aldrig hem,Mozby,1 ContributorDen Eviga Vilan LyricsEn stadig g...,0,0,vi ska aldrig hem
439,LAVISH 2.0,PERZZONA,8 ContributorsScarface Script LyricsScarface\n...,0,0,lavish 2.0


In [1215]:
trainbus_18 = fetch_lyrics(api, track_name = "99 Problems", artist_name = "JAY-Z")

Searching for "99 Problems" by JAY-Z...
Done.


In [1216]:
trainbus_27 = fetch_lyrics(api, track_name = "My Name Is", artist_name = "Eminem")

Searching for "My Name Is" by Eminem...
Done.


In [1217]:
trainbus_29 = fetch_lyrics(api, track_name = "Independent Women Part I", artist_name = "Destiny's Child")

Searching for "Independent Women Part I" by Destiny's Child...
Done.


In [1218]:
trainbus_234 = fetch_lyrics(api, track_name = "All 4 Nothing (I’m So In Love)", artist_name = "Lauv")

Searching for "All 4 Nothing (I’m So In Love)" by Lauv...
Done.


In [1220]:
nextt.loc[18, 'lyrics'] = trainbus_18
nextt.loc[27, 'lyrics'] = trainbus_27
nextt.loc[29, 'lyrics'] = trainbus_29
nextt.loc[234, 'lyrics'] = trainbus_234

In [1221]:
trainBus.loc[nextt.index, 'lyrics'] = nextt['lyrics']

# Save Cleaned Lyrics Data

In [1222]:
trainBus.to_csv('trainbus_correct_lyrics.csv', index=False)
library.to_csv('library_correct_lyrics.csv', index=False)
mall.to_csv('mall_correct_lyrics.csv', index=False)
bar.to_csv('bar_correct_lyrics.csv', index=False)
gym.to_csv('gym_correct_lyrics.csv', index=False)
bedroom.to_csv('bedroom_correct_lyrics.csv', index=False)
office.to_csv('office_correct_lyrics.csv', index=False)
restaurant.to_csv('restaurant_correct_lyrics.csv', index=False)